In [ ]:
import pandas as pd
from sklearn.utils import shuffle
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
import warnings
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import shap
import xgboost as xgb
from tqdm import tqdm, tqdm_notebook
import datetime
import numpy as np

In [ ]:
seed = 15
np.random.seed(seed)

In [ ]:
esa_21_22_corners_parsed = pd.read_pickle('data/corners_xgb_esa_21_22_0912.pkl')
esa_22_23_corners_parsed = pd.read_pickle('data/corners_xgb_esa_22_23_0912.pkl')
euro_corners_parsed = pd.read_pickle('data/corners_xgb_euro_0912.pkl')
fifa_corners_parsed = pd.read_pickle('data/corners_xgb_fifa_2022_0912.pkl')

esa_21_22_core = pd.read_pickle('data/38_108.pkl')
esa_22_23_core = pd.read_pickle('data/38_235.pkl')

In [ ]:
label = 'was_shot'
# label = 'shot_on_goal'

Get test set from ESA 22/23

In [ ]:
def get_test_set(df, df_core):
    dfs_test = []
    dfs_train = []
    teams = list(set(df['team'].values))
    test_matches = df_core[esa_22_23_core['match_week'] >= 18]['match_id'].values
    train_matches = df_core[esa_22_23_core['match_week'] < 18]['match_id'].values
    for team in teams:
        for match_id in test_matches:
            df_result = df[(df['team'] == team) & (df['match_id'] == match_id)]
            dfs_test.append(df_result)
        for match_id in train_matches:
            df_result = df[(df['team'] == team) & (df['match_id'] == match_id)]
            dfs_train.append(df_result)
    df_test = pd.concat(dfs_test)
    df_train = pd.concat(dfs_train)
    return df_test, df_train


esa_22_23_test, esa_22_23_train = get_test_set(esa_22_23_corners_parsed, esa_22_23_core)
print(f'Length of 22/23 test set: {len(esa_22_23_test)}')
print(f'Length of 22/23 train set: {len(esa_22_23_train)}')

Concat train set and name test set properly

In [ ]:
test_set = esa_22_23_test
train_set = pd.concat([esa_22_23_train, esa_21_22_corners_parsed, euro_corners_parsed, fifa_corners_parsed])

print(f'Length of test set: {len(test_set)}')
print(f'Length of train set: {len(train_set)}')

print(f'Length of esa_22_23_test: {len(esa_22_23_test)}')
print(f'Length of esa_22_23_train: {len(esa_22_23_train)}')
print(f'Length of esa_21_22_corners_parsed: {len(esa_21_22_corners_parsed)}')
print(f'Length of euro_corners_parsed: {len(euro_corners_parsed)}')
print(f'Length of fifa_corners_parsed: {len(fifa_corners_parsed)}')


Prepare labels and data balance

In [ ]:
def prepare_df_with_label(train_set, test_set, label):
    if label == 'was_shot':
        columns_to_drop = ['id','match_id', 'team', 'corner_score', 'obv', 'xg', 'shot_on_goal', 'num_of_shot']
        test_set['label'] = [1 if (shots > 0) else 0 for shots in test_set['num_of_shot'].values]
        train_set['label'] = [1 if (shots > 0) else 0 for shots in train_set['num_of_shot'].values]

    elif label == 'shot_on_goal':
        columns_to_drop = ['id','team', 'corner_score', 'obv', 'xg', 'num_of_shot', 'shot_on_goal', 'match_id']
        test_set['label'] = [1 if (shots > 0) else 0 for shots in test_set['shot_on_goal'].values]
        train_set['label'] = [1 if (shots > 0) else 0 for shots in train_set['shot_on_goal'].values]

    else:
        raise ValueError('Incorrect name of label!')
    
    print(f'Number of {label} in test set: {sum(test_set["label"].values)}')
    print(f'Number of {label} in train set: {sum(train_set["label"].values)}')

    num_of_positive = sum(test_set["label"].values) +sum(train_set["label"].values)
    num_of_negative = len(test_set) + len(train_set) - num_of_positive

    print(f'Number of positive in whole set: {num_of_positive}')
    print(f'Number of negative in whole set: {num_of_negative}')

    data_balance = num_of_negative / num_of_positive
    print(f'Data balance: {data_balance}')

    return train_set, test_set, columns_to_drop, data_balance

train_set, test_set, columns_to_drop, data_balance = prepare_df_with_label(train_set, test_set, label)

In [ ]:
columns_to_drop

In [ ]:
def get_sets_for_model(df_train, df_test):
    df_train = shuffle(df_train)
    df_test = shuffle(df_test)

    df_train_columns = df_train.columns.sort_values()
    df_test_columns = df_test.columns.sort_values()

    df_train = pd.DataFrame(df_train, columns=df_train_columns)
    df_test = pd.DataFrame(df_test, columns=df_test_columns)

    y_train = df_train['label']
    X_train = df_train.drop(columns=['label'])

    y_test = df_test['label']
    X_test = df_test.drop(columns=['label'])

    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = get_sets_for_model(train_set, test_set)

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split

In [ ]:
def find_proper_parameters(X_train, X_test, y_train, columns_to_drop, data_balance):
    model = XGBClassifier(objective='binary:logistic', scale_pos_weight=data_balance)
    input_X_train = X_train.drop(columns=columns_to_drop)
    model.fit(input_X_train, y_train)

    input_X_test = X_test.drop(columns=columns_to_drop)
    XGB_predictions = model.predict(input_X_test)

    param_grid = {
        'colsample_bytree': [0.9, 1.0],
        'learning_rate': [0.01, 0.1],
        'max_depth': [1, 2, 3, 4],
        'n_estimators': [100, 200, 300, 400],
        'subsample': [0.7, 0.8, 0.9],
        'gamma' : [0, 0.01, 0.001],
        'reg_alpha' : [0, 0.01, 0.001]
    }

    grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='f1', cv=3, verbose=10)
    grid_search.fit(input_X_train, y_train)
    return grid_search.best_params_

# proper_params = find_proper_parameters(X_train, X_test, y_train, columns_to_drop, data_balance)

# now = datetime.datetime.now()
# proper_params = pd.DataFrame(proper_params, index=[0])
# proper_params.to_csv(f'proper_params_{label}_{now.strftime("%Y-%m-%d")}.csv')

In [ ]:
with open (f'proper_params_{label}_2023-09-26.csv', 'r') as f:
    proper_params = pd.read_csv(f, index_col=0)


In [ ]:
proper_params

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split

proper_params = proper_params.to_dict('records')[0]

def train_model(X_train, X_test, y_train, columns_to_drop, data_balance, parameters):
    model = XGBClassifier(objective='binary:logistic', scale_pos_weight=data_balance, **parameters)
    input_X_train = X_train.drop(columns=columns_to_drop)
    model.fit(input_X_train, y_train)
    return model

model = train_model(X_train, X_test, y_train, columns_to_drop, data_balance, proper_params)

In [ ]:
def get_predictions(X_test, columns_to_drop, model):
    input_X_test = X_test.drop(columns=columns_to_drop)
    XGB_predictions = model.predict(input_X_test)
    return XGB_predictions

predictions = get_predictions(X_test, columns_to_drop, model)

In [ ]:
def get_confusion_matrix(y_test, predictions):
    tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()
    print(f'TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}')
    return tn, fp, fn, tp

tn, fp, fn, tp = get_confusion_matrix(y_test, predictions)

In [ ]:
def evaluate_model(y_test, pred):
    print(f'Accuracy: {accuracy_score(y_test, pred)}')
    print(f'Precision: {precision_score(y_test, pred)}')
    print(f'Recall: {recall_score(y_test, pred)}')
    print(f'F1: {f1_score(y_test, pred)}') 

evaluate_model(y_test, predictions)

In [ ]:
def generate_scores(model, X_test):
    input_X_test = X_test.drop(columns=columns_to_drop)
    model_proba = model.predict_proba(input_X_test)
    X_test['success_probability'] = [m[1] for m in model_proba]
    return X_test

results = generate_scores(model, X_test)

In [ ]:
def print_correlation(df_with_scores, test_set):
    corr_with_xg = df_with_scores['success_probability'].corr(df_with_scores['xg'])
    corr_with_obv = df_with_scores['success_probability'].corr(df_with_scores['obv'])
    corr_with_corner_score = df_with_scores['success_probability'].corr(df_with_scores['corner_score'])
    print(f'Correlation with xg: {corr_with_xg:.3f}')
    print(f'Correlation with obv: {corr_with_obv:.3f}')
    print(f'Correlation with corner_score: {corr_with_corner_score:.3f}')
    # return df_with_scores

print_correlation(results, test_set)

In [ ]:
def get_avg_metrics_for_each_team(df_with_scores, test_set):
    df_with_scores = df_with_scores.groupby(['team']).agg({'success_probability': 'mean', 'obv': 'mean', 'xg': 'mean', 'corner_score' : 'mean', 'num_of_shot' : 'mean', 'shot_on_goal' : 'mean'}).reset_index()
    df_with_scores['total_corners'] = df_with_scores['team'].map(test_set['team'].value_counts()) / 18
    df_with_scores = df_with_scores.sort_values(by=['success_probability'], ascending=False)
    return df_with_scores.round(3)

# def get_avg_metrics_for_each_team(df_with_scores, test_set):
#     df_with_scores = df_with_scores.groupby(['team']).agg({'success_probability': 'mean', 'obv': 'mean', 'xg': 'mean', 'corner_score' : 'mean', 'num_of_shot' : 'mean', 'shot_on_goal' : 'mean'}).reset_index()
#     df_with_scores['total_corners'] = df_with_scores['team'].map(test_set['team'].value_counts()) / 18
#     # was_shot column with corners which has num_of_shot > 0 else 0
#     df_with_scores['was_shot'] = df_with_scores['team'].map(test_set[test_set['num_of_shot'] > 0]['team'].value_counts()) / 18

#     df_with_scores = df_with_scores.sort_values(by=['success_probability'], ascending=False)
#     return df_with_scores.round(3)

def get_sum_metrics_for_each_team(df_with_scores, test_set):
    df_with_scores = df_with_scores.groupby(['team']).agg({'success_probability': 'sum', 'obv': 'sum', 'xg': 'sum', 'corner_score' : 'sum', 'num_of_shot' : 'sum', 'shot_on_goal' : 'sum'}).reset_index()
    df_with_scores['total_corners'] = df_with_scores['team'].map(test_set['team'].value_counts())
    df_with_scores = df_with_scores.sort_values(by=['success_probability'], ascending=False)
    return df_with_scores.round(3)

def get_med_metrics_for_each_team(df_with_scores, test_set):
    df_with_scores = df_with_scores.groupby(['team']).agg({'success_probability': 'median', 'obv': 'median', 'xg': 'median', 'corner_score' : 'median'}).reset_index()
    df_with_scores = df_with_scores.sort_values(by=['success_probability'], ascending=False)
    return df_with_scores.round(3)

res_avg = get_avg_metrics_for_each_team(results, test_set)
res_sum = get_sum_metrics_for_each_team(results, test_set)
res_med = get_med_metrics_for_each_team(results, test_set)

In [ ]:
res_avg.to_csv(f'final/res_avg_{label}_2023-09-26.csv')

In [ ]:
X_train, X_test, y_train, y_test = get_sets_for_model(train_set, test_set)

In [ ]:
X_train.columns

In [ ]:
def generate_feature_importances(model, X_test, columns_to_drop):
    results = pd.DataFrame()
    input_X_test = X_test.drop(columns=columns_to_drop)
    explainer = shap.TreeExplainer(model)

    data_whole = xgb.DMatrix(input_X_test)
    shap_values_whole = explainer.shap_values(data_whole) 
    mean_importance_whole = shap_values_whole.mean(axis=0)

    results['feature'] = input_X_test.columns
    results['importance'] = mean_importance_whole.round(3)
    return results.sort_values(by='importance', ascending=False)

In [ ]:
feature_importances = generate_feature_importances(model, X_test, columns_to_drop)

In [ ]:
feature_importances.sort_values(by='importance', ascending=False).to_csv(f'final/feature_importances_{label}_2023-09-26.csv')

In [ ]:
def generate_feature_importances_per_team(model, X_test, columns_to_drop):
    input_X_test = X_test.drop(columns=columns_to_drop)

    explainer = shap.TreeExplainer(model)
    column_to_group_by = 'team'
    groups = X_test.groupby(column_to_group_by)
    features_importance_per_group = {}

    for group_name, group_data in groups:
        results = pd.DataFrame()
        data = xgb.DMatrix(group_data.drop(columns=columns_to_drop))
        shap_values = explainer.shap_values(data)
        mean_importance = shap_values.mean(axis=0)

        column_indices = group_data.drop(columns=columns_to_drop).columns
        results['feature'] = column_indices
        results['importance'] = mean_importance.round(3)
        features_importance_per_group[group_name] = results.sort_values(by='importance', ascending=False)

    return features_importance_per_group

In [ ]:
features_importance_per_team = generate_feature_importances_per_team(model, X_test, columns_to_drop)

In [ ]:
# features_importance_per_team['Zagłębie Lubin'].to_csv(f'final/feat_imp_{label}_Zagłębie_Lubin_2023-09-26.csv')

In [ ]:
# features_importance_per_team['RKS Radomiak Radom'].to_csv(f'final/feat_imp_{label}_RKS_Radomiak_Radom_2023-09-26.csv')

In [ ]:
features_importance_per_team['Warta Poznań'].to_csv(f'final/feat_imp_{label}_Warta_Poznań_2023-09-26.csv')

In [ ]:
features_importance_per_team['Lech Poznan'].to_csv(f'final/feat_imp_{label}_Lech_Poznan_2023-09-26.csv')

In [ ]:
features_importance_per_team['Legia Warszawa'].to_csv(f'final/feat_imp_{label}_Legia_Warszawa_2023-09-26.csv')

In [ ]:
features_importance_per_team['Wisła Płock'].to_csv(f'final/feat_imp_{label}_Wisła_Płock_2023-09-26.csv')